# IceCube: From Raw Data to Machine Learning
## A Step-by-Step 2-Hour Session for PhD Students

---

### What we will do today

We have a real particle physics detector at the South Pole called **IceCube**. Every time a neutrino passes through the ice, it creates a flash of blue light that our sensors detect. Our job is to figure out **which direction** that neutrino came from — just from the pattern of light flashes.

We will go through this in five steps:

```
STEP 1 → Load the data and understand what we have
STEP 2 → Visualise events — see what a neutrino looks like
STEP 3 → Traditional method (LineFit) — the physics approach
STEP 4 → Machine learning (GNN) — the modern approach
STEP 5 → Compare them head-to-head
```

**You only need to press Shift+Enter on each cell, in order.**

---

### The two files we are working with

| File | What it contains |
|------|------------------|
| `train_meta.parquet` | One row per neutrino event. Tells us the **true direction** (azimuth + zenith). |
| `batch_1.parquet` | One row per sensor flash. Tells us **which sensor fired, when, and how bright**. |

Think of it this way:
- `train_meta` = the answer sheet (true neutrino directions)
- `batch_1` = the raw detector data (sensor hits we must interpret)

---
## STEP 1 — Load the Data

We load both files and explore what is inside them.

**Run the cells below one by one.**

### Cell 1.1 — Import the tools we need

These are standard Python libraries. Nothing IceCube-specific yet.

In [ ]:
import numpy as np                 # numbers and arrays
import pandas as pd                # tables (like Excel in Python)
import pyarrow.parquet as pq       # read .parquet files
import matplotlib.pyplot as plt    # make plots
import time                        # measure how fast things run
import os, warnings
warnings.filterwarnings("ignore")

print("All libraries loaded successfully!")
print("NumPy version  :", np.__version__)
print("Pandas version :", pd.__version__)


### Cell 1.2 — Load `train_meta.parquet` (the label file)

This file has one row per neutrino event. It tells us the **true direction** of each neutrino — this is what we are trying to predict.

- `azimuth` = angle around the compass [0 to 2π radians]
- `zenith`  = angle from straight down [0 = downward, π = upward]
- `batch_id` = which batch file contains the sensor hits for this event

In [ ]:
# Open and read the meta file
with open("train_meta.parquet", "rb") as f:
    meta = pq.read_table(f).to_pandas()

print("Shape:", meta.shape, "  ← rows × columns")
print()
print("Column names:", list(meta.columns))
print()
print("First 5 rows:")
meta.head()


### Cell 1.3 — Load `batch_1.parquet` (the sensor hit file)

This file has one row **per sensor flash** (called a hit).

**Important:** `event_id` is stored as the file index — we call `.reset_index()` to bring it back as a regular column.

Each row tells us:
- `sensor_id` → which of the 5,160 sensors detected light
- `time`      → when it detected light [nanoseconds]
- `charge`    → how much light it detected [photoelectrons]
- `auxiliary` → is this a noisy hit we should ignore? (True = ignore)

In [ ]:
# Open and read the batch file
# .reset_index() brings event_id back from the file index into a column
with open("batch_1.parquet", "rb") as f:
    batch = pq.read_table(f).to_pandas().reset_index()

print("Shape:", batch.shape, "  ← rows × columns")
print()
print("Column names:", list(batch.columns))
print()
print("First 8 rows:")
batch.head(8)


### Cell 1.4 — Remove noisy hits

About 22% of hits have `auxiliary = True`. These are low-quality hits (noise, or weakly triggered sensors). We remove them to keep only the reliable signal hits.

This is the first data-cleaning step — equivalent to what the IceCube experiment calls **SeededRT cleaning**.

In [ ]:
# Keep only high-quality (HLC) hits — remove auxiliary noise hits
batch_clean = batch[batch["auxiliary"] == False].copy()

print(f"Raw hits       : {len(batch):,}")
print(f"After cleaning : {len(batch_clean):,}")
print(f"Removed        : {len(batch)-len(batch_clean):,}  "
      f"({(len(batch)-len(batch_clean))/len(batch):.1%} of hits)")


### Cell 1.5 — Join sensor hits with labels

Right now, `batch_clean` has sensor hits but no direction labels. And `meta` has labels but no sensor hits.

We join them using the `event_id` column — like a SQL join — so each sensor hit knows the true direction of its neutrino.

In [ ]:
# Get only the batch_1 events from meta (using batch_id == 1)
meta_b1 = meta[meta["batch_id"] == 1][["event_id", "azimuth", "zenith"]]

# Join on event_id
df = batch_clean.merge(meta_b1, on="event_id", how="inner")

print("After joining:")
print(f"  Rows    : {len(df):,}  (one row per sensor hit)")
print(f"  Events  : {df['event_id'].nunique():,}  (unique neutrino interactions)")
print()
print("Sample (5 hits from one event):")
df[df["event_id"] == df["event_id"].iloc[0]].head(5)[
    ["event_id","sensor_id","time","charge","azimuth","zenith"]]


### Cell 1.6 — Basic statistics of the dataset

Let us look at how many sensor hits each event produces. This varies a lot — from just a handful to tens of thousands — because neutrino energies span many orders of magnitude.

In [ ]:
# Count how many hits each event has
hits_per_event = df.groupby("event_id").size()

print("Hits per event:")
print(f"  Minimum  : {hits_per_event.min()}")
print(f"  Median   : {hits_per_event.median():.0f}")
print(f"  Mean     : {hits_per_event.mean():.0f}")
print(f"  Maximum  : {hits_per_event.max():,}")
print()
print("This heavy tail (median=20, max=45,913) tells us:")
print("  → Most events are low-energy (few hits)")
print("  → Some events are very high energy (thousands of hits)")
print("  → Any ML model must handle variable-size inputs")


### Cell 1.7 — Visualise the dataset distributions

Four plots showing the shape of the data:
1. How many hits per event (log scale — very skewed!)
2. Azimuth of true neutrino direction (should be uniform)
3. Zenith of true neutrino direction
4. Charge per hit (most hits are small; a few are very bright)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Plot 1: hits per event
ax = axes[0]
ax.hist(np.log10(hits_per_event + 1), bins=40,
        color="steelblue", edgecolor="k", linewidth=0.4)
ax.set_xlabel("log₁₀(hits per event)", fontsize=11)
ax.set_ylabel("Number of events", fontsize=11)
ax.set_title("Hit multiplicity (log scale — heavy tail!)", fontsize=11)
ax.grid(alpha=0.3)
# Add reference lines
for logn, lbl in [(1,"10"),(2,"100"),(3,"1000")]:
    ax.axvline(logn, ls="--", color="gray", lw=1, alpha=0.7)
    ax.text(logn+0.05, ax.get_ylim()[1]*0.9, lbl, fontsize=8, color="gray")

# Plot 2: azimuth
ax = axes[1]
ax.hist(meta_b1["azimuth"], bins=60, color="darkorange", edgecolor="k", lw=0.4)
ax.set_xlabel("Azimuth φ [radians]", fontsize=11)
ax.set_title("True neutrino azimuth (uniform → detector is isotropic)", fontsize=11)
ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax.set_xticklabels(["0","π/2","π","3π/2","2π"])
ax.grid(alpha=0.3)

# Plot 3: zenith
ax = axes[2]
ax.hist(meta_b1["zenith"], bins=60, color="crimson", edgecolor="k", lw=0.4)
ax.axvline(np.pi/2, ls="--", color="k", lw=2, label="π/2 = horizon")
ax.set_xlabel("Zenith θ [radians]", fontsize=11)
ax.set_title("True neutrino zenith (0=downward, π=upward)", fontsize=11)
ax.set_xticks([0, np.pi/2, np.pi])
ax.set_xticklabels(["0 (down)","π/2 (horizon)","π (up)"])
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Plot 4: charge
ax = axes[3]
ax.hist(np.log10(df["charge"] + 0.01), bins=50,
        color="mediumpurple", edgecolor="k", lw=0.4)
ax.set_xlabel("log₁₀(charge [PE])", fontsize=11)
ax.set_title("Hit charge distribution (most hits are small)", fontsize=11)
ax.grid(alpha=0.3)

plt.suptitle("IceCube batch_1 — Dataset Overview", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("1_dataset_overview.pdf", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved as 1_dataset_overview.pdf")


---
## STEP 2 — Visualise Events in the Detector

Before we can reconstruct direction, we need to understand **where** each sensor is in space. Then we can plot events as 3D point clouds and see what they look like.

### Cell 2.1 — Load the sensor geometry

Each `sensor_id` (0–5159) corresponds to a physical location $(x, y, z)$ in the ice, measured in metres.

**You already have `sensor_geometry.csv` in your folder — just set the correct path below.**

Change `GEO_PATH` in the next cell to wherever the file lives on your machine. For example:

```python
GEO_PATH = "sensor_geometry.csv"          # same folder as the notebook
GEO_PATH = "/data/icecube/sensor_geometry.csv"  # absolute path
GEO_PATH = "../data/sensor_geometry.csv"  # relative path
```

The cell will load the real geometry if the file is found, or fall back to an approximate version if not — with a clear message telling you which one is active.

In [ ]:
# ── Set the path to sensor_geometry.csv ──────────────────────────────────────
# Change this to wherever your file is located.
# Examples:
#   GEO_PATH = "sensor_geometry.csv"                    # same folder as notebook
#   GEO_PATH = "/home/user/data/sensor_geometry.csv"   # absolute path
#   GEO_PATH = "../datasets/sensor_geometry.csv"        # relative path

GEO_PATH = "sensor_geometry.csv"   # ← CHANGE THIS IF NEEDED

# ── Load geometry ─────────────────────────────────────────────────────────────
if os.path.exists(GEO_PATH):
    geo = pd.read_csv(GEO_PATH).sort_values("sensor_id")
    SENSOR_XYZ = geo[["x","y","z"]].values.astype(np.float32)
    GEO_SOURCE = f"Real geometry loaded from: {GEO_PATH}"
    print("Loaded real sensor geometry!")
    print()
    print("First 5 rows of the geometry file:")
    print(geo.head())

else:
    # Approximate geometry — reproduces the IceCube layout
    # (only used if the real file is not found)
    print(f"File not found at: {GEO_PATH}")
    print("Building approximate geometry instead...")
    print("(Set GEO_PATH correctly above to use the real geometry)")
    print()

    def hex_grid(n_rings, spacing):
        pts = [(0., 0.)]
        for ring in range(1, n_rings + 1):
            for side in range(6):
                a = np.radians(60 * side)
                for s in range(ring):
                    x = ring * np.cos(a) + s * np.cos(a + np.radians(120))
                    y = ring * np.sin(a) + s * np.sin(a + np.radians(120))
                    pts.append((x * spacing, y * spacing))
        return np.array(pts)

    std_xy = hex_grid(5, 125.)[:78]
    dc_xy  = hex_grid(2,  72.)[:8]

    rows = []
    for sx, sy in std_xy:
        for z in np.linspace(-500., -500. - 17. * 59, 60):
            rows.append([sx, sy, z])
    for sx, sy in dc_xy:
        for z in np.linspace(-150., -150. - 7. * 59, 60):
            rows.append([sx, sy, z])

    SENSOR_XYZ = np.array(rows, dtype=np.float32)[:5160]
    GEO_SOURCE = "Approximate geometry (sensor_geometry.csv not found)"

print()
print(f"Status : {GEO_SOURCE}")
print(f"Shape  : {SENSOR_XYZ.shape}  ← (5160 sensors) × (x, y, z)")
print(f"x range: [{SENSOR_XYZ[:,0].min():.0f}, {SENSOR_XYZ[:,0].max():.0f}] m")
print(f"y range: [{SENSOR_XYZ[:,1].min():.0f}, {SENSOR_XYZ[:,1].max():.0f}] m")
print(f"z range: [{SENSOR_XYZ[:,2].min():.0f}, {SENSOR_XYZ[:,2].max():.0f}] m")
print()
if "Real geometry" in GEO_SOURCE:
    print("Great — real geometry loaded. LineFit will work correctly.")
else:
    print("WARNING: Using approximate geometry.")
    print("LineFit angular errors will be ~58 deg instead of ~20 deg.")
    print("Set GEO_PATH to the correct path and re-run this cell.")

### Cell 2.2 — Plot the detector geometry

Let us visualise where the 5,160 sensors sit in 3D space. This helps students understand the scale and layout of IceCube.

IceCube is 1 km × 1 km × 1 km — buried 1.5 to 2.5 km below the South Pole surface.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 6))

# 3D view of all sensors
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.scatter(SENSOR_XYZ[:,0], SENSOR_XYZ[:,1], SENSOR_XYZ[:,2],
            s=1.5, c=SENSOR_XYZ[:,2], cmap="plasma", alpha=0.5)
ax1.set_xlabel("x [m]"); ax1.set_ylabel("y [m]"); ax1.set_zlabel("z [m]")
ax1.set_title("IceCube sensor positions (5160 sensors, 3D view)", fontsize=11)

# Top-down view (x-y plane)
ax2 = fig.add_subplot(1, 2, 2)
ax2.scatter(SENSOR_XYZ[::60, 0], SENSOR_XYZ[::60, 1],
            s=25, color="steelblue", alpha=0.7, label="Standard strings (78)")
dc_mask = np.arange(78*60, 86*60, 60)   # DeepCore string positions
ax2.scatter(SENSOR_XYZ[dc_mask, 0], SENSOR_XYZ[dc_mask, 1],
            s=60, color="crimson", marker="*", label="DeepCore strings (8)")
ax2.set_xlabel("x [m]"); ax2.set_ylabel("y [m]")
ax2.set_title("Top-down view (one dot per string)", fontsize=11)
ax2.set_aspect("equal"); ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.suptitle("IceCube Detector Geometry", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("2_detector_geometry.pdf", dpi=150, bbox_inches="tight")
plt.show()

print("The detector occupies roughly a 1 km × 1 km × 1 km cube of Antarctic ice.")
print("Standard string spacing: ~125 m")
print("DeepCore (inner) strings: ~70 m spacing, for lower energy neutrinos")


### Cell 2.3 — Pick three events to display

We choose one low-energy event, one typical event, and one high-energy event. This shows students that the data looks very different across the energy range.

In [ ]:
# Categorise events by hit count (proxy for energy)
small_events  = hits_per_event[hits_per_event.between(15,  30)].index
medium_events = hits_per_event[hits_per_event.between(60,  90)].index
large_events  = hits_per_event[hits_per_event.between(150, 300)].index

# Pick one of each
eid_small  = small_events[0]
eid_medium = medium_events[0]
eid_large  = large_events[0]

# Show what they look like in the table
for label, eid in [("Low energy (few hits)",    eid_small),
                   ("Typical event",             eid_medium),
                   ("High energy (many hits)",   eid_large)]:
    hits = df[df["event_id"] == eid]
    az   = float(hits["azimuth"].iloc[0])
    ze   = float(hits["zenith"].iloc[0])
    print(f"{label}:")
    print(f"  event_id = {eid}")
    print(f"  N hits   = {len(hits)}")
    print(f"  azimuth  = {az:.3f} rad")
    print(f"  zenith   = {ze:.3f} rad  "
          f"({'upgoing' if ze > np.pi/2 else 'downgoing'})")
    print()


### Cell 2.4 — 3D event display function

We define a reusable function that takes any event and plots it:
- Each sphere = one sensor that detected light
- **Size** of sphere ∝ charge (brightness)
- **Colour** = arrival time (yellow = early, purple = late)
- **Red dashed arrow** = true neutrino direction

In [ ]:
def plot_event_3d(event_id, df, sensor_xyz, ax, title=""):
    """
    Draw one IceCube event in 3D.
    
    - Spheres show which sensors fired
    - Size  = charge (how much light)
    - Colour = time  (yellow=early, purple=late)
    - Red dashed line = true neutrino direction
    """
    hits = df[df["event_id"] == event_id].copy()
    sid  = hits["sensor_id"].values.astype(int)
    t    = hits["time"].values.astype(float)
    q    = hits["charge"].values.astype(float)
    pos  = sensor_xyz[sid]   # look up (x,y,z) for each sensor

    # Normalise time to [0,1] for colouring
    t_norm = (t - t.min()) / (t.max() - t.min() + 1.)

    # Size: sqrt of charge (so very bright DOMs don't dominate visually)
    sizes = 15 + np.sqrt(np.clip(q, 0, np.percentile(q, 95))) * 50

    # Draw the hits
    sc = ax.scatter(pos[:,0], pos[:,1], pos[:,2],
                    c=t_norm, cmap="plasma_r", s=sizes,
                    alpha=0.85, edgecolors="none", zorder=5)

    # Draw the true direction arrow through the charge-weighted centre
    az  = float(hits["azimuth"].iloc[0])
    ze  = float(hits["zenith"].iloc[0])
    d   = np.array([np.sin(ze)*np.cos(az),
                    np.sin(ze)*np.sin(az),
                    np.cos(ze)])
    cog = np.average(pos, weights=q + 0.01, axis=0)   # centre of light
    t_line = np.linspace(-250, 250, 50)
    line   = cog + t_line[:,None] * d
    ax.plot(line[:,0], line[:,1], line[:,2],
            "r--", lw=2.5, alpha=0.8, label="True direction")

    ax.set_xlabel("x [m]", fontsize=8)
    ax.set_ylabel("y [m]", fontsize=8)
    ax.set_zlabel("z [m]", fontsize=8)
    direction_word = "upgoing" if ze > np.pi/2 else "downgoing"
    ax.set_title(f"{title} event_id = {event_id} "
                 f"{len(hits)} hits   {direction_word} "
                 f"az={az:.2f}  ze={ze:.2f} rad", fontsize=9)
    ax.legend(fontsize=7)
    return sc

print("Event display function defined — ready to plot!")


### Cell 2.5 — Draw the three events side by side

**What to look for:**
- A **track-like event** (muon neutrino) shows light arriving in a sequence — the colour sweeps from yellow to purple along one direction
- A **cascade-like event** (electron neutrino) shows a compact blob — all sensors firing at almost the same time
- More hits = more light = higher energy

In [ ]:
fig = plt.figure(figsize=(18, 7))

for col, (eid, title) in enumerate([
    (eid_small,  "Low energy"),
    (eid_medium, "Typical event"),
    (eid_large,  "High energy"),
]):
    ax = fig.add_subplot(1, 3, col+1, projection="3d")
    sc = plot_event_3d(eid, df, SENSOR_XYZ, ax, title=title)
    plt.colorbar(sc, ax=ax, label="Arrival time (yellow=early, purple=late)",
                 shrink=0.5, pad=0.12, orientation="horizontal")

plt.suptitle("IceCube Event Displays "
             "Sphere size ∝ charge (brightness)   Colour ∝ arrival time",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("3_event_displays.pdf", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 3_event_displays.pdf")


### Cell 2.6 — The time vs z plot (easier to read)

The 3D plot is beautiful but hard to interpret. A simpler and more powerful view is **time vs z-position**.

If a neutrino travels mostly vertically:
- A **downgoing track** makes a diagonal line sloping right (later time at lower z)
- A **upgoing track** makes a diagonal line sloping left
- A **cascade** makes a vertical cluster (all sensors fire at the same time)

The slope of the line tells us the speed — which should be roughly $c_{\rm ice} = 0.227$ m/ns.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, eid, title in zip(axes,
    [eid_small, eid_medium, eid_large],
    ["Low energy", "Typical event", "High energy"]):

    hits = df[df["event_id"] == eid]
    sid  = hits["sensor_id"].values.astype(int)
    t    = hits["time"].values.astype(float)
    q    = hits["charge"].values.astype(float)
    z    = SENSOR_XYZ[sid, 2]

    sc = ax.scatter(t, z,
                    s  = np.sqrt(np.clip(q, 0, np.percentile(q, 95))) * 35 + 8,
                    c  = t,
                    cmap = "plasma_r",
                    alpha = 0.8,
                    edgecolors = "k",
                    linewidths = 0.3)
    plt.colorbar(sc, ax=ax, label="Hit time [ns]", shrink=0.75)

    az = float(hits["azimuth"].iloc[0])
    ze = float(hits["zenith"].iloc[0])
    direction_word = "upgoing" if ze > np.pi/2 else "downgoing"

    ax.set_xlabel("Hit time [ns]", fontsize=11)
    ax.set_ylabel("Sensor z position [m]", fontsize=11)
    ax.set_title(f"{title}  (event {eid}) "
                 f"{len(hits)} hits | {direction_word} "
                 f"az={az:.2f} ze={ze:.2f} rad", fontsize=10)
    ax.grid(alpha=0.3)

plt.suptitle("Time vs z-position: the wavefront of Cherenkov light "
             "A clear diagonal = track-like.   A vertical cluster = cascade-like.",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("4_time_vs_z.pdf", dpi=150, bbox_inches="tight")
plt.show()

print("Key insight:")
print("  • Diagonal pattern → sensors fire in sequence → track (directional)")
print("  • Vertical cluster  → sensors fire together  → cascade (spherical)")
print()
print("The slope of the diagonal ≈ 1/(c_ice) = 1/0.227 = 4.4 ns/m")


---
## STEP 3 — Traditional Reconstruction: LineFit

Now we try to **reconstruct the neutrino direction** using a purely physics-based method. No machine learning yet.

**The idea:** if photons travel in straight lines at the speed of light in ice, then the arrival time at each sensor depends linearly on how far the sensor is along the neutrino's path. We find the direction that best explains the observed timing pattern — using linear regression.

### Cell 3.1 — Understanding the geometry problem

To run LineFit, we need to know where each sensor is in 3D space. The `sensor_id` column gives us an integer 0–5159, but we need $(x,y,z)$ in metres.

This mapping lives in `sensor_geometry.csv` on Kaggle. **Without it, LineFit gives wrong results** — we show this explicitly below so students understand exactly what the geometry file does.

In [ ]:
# Let us check whether our geometry gives physically sensible results.
# A simple test: for a sample event, the correlation between hit time and
# sensor position projected onto the TRUE direction should be strong.
# If the geometry is wrong, this correlation will be weak.

test_eid  = eid_medium
test_hits = df[df["event_id"] == test_eid]
sid_t = test_hits["sensor_id"].values.astype(int)
t_t   = test_hits["time"].values.astype(float)

# True direction unit vector
az_t = float(test_hits["azimuth"].iloc[0])
ze_t = float(test_hits["zenith"].iloc[0])
d_true = np.array([np.sin(ze_t)*np.cos(az_t),
                   np.sin(ze_t)*np.sin(az_t),
                   np.cos(ze_t)])

# Project sensor positions onto the true direction
pos_t = SENSOR_XYZ[sid_t]
proj  = pos_t @ d_true   # dot product = how far along the track

# Correlation between time and projection
corr = np.corrcoef(t_t, proj)[0, 1]

print(f"Test event {test_eid}:")
print(f"  Number of hits : {len(test_hits)}")
print(f"  True direction : az={az_t:.3f}  ze={ze_t:.3f} rad")
print()
print(f"  Correlation(time, projection onto true direction) = {corr:.3f}")
print()
if abs(corr) > 0.6:
    print("  ✓ Strong correlation! Geometry is consistent with timing.")
    print("    LineFit should work well.")
elif abs(corr) > 0.3:
    print("  ⚠ Moderate correlation. Geometry may be approximate.")
    print("    LineFit will give partial results.")
else:
    print("  ✗ Weak correlation. Geometry is inconsistent with timing.")
    print("    This means sensor_id → (x,y,z) mapping is wrong.")
    print("    LineFit will give poor results — we need sensor_geometry.csv!")


### Cell 3.2 — Implement LineFit from scratch

LineFit is a **weighted linear regression** of sensor position on hit time.

Mathematically, we minimise:
$$\chi^2 = \sum_i q_i \left(t_i^{\rm obs} - t_i^{\rm expected}\right)^2$$

where $q_i$ is the charge (weight), $t_i^{\rm obs}$ is when sensor $i$ fired, and $t_i^{\rm expected}$ depends on the track direction we are estimating.

The **closed-form solution** is:
$$\hat{d} \propto \frac{\sum_i q_i \,(\vec{r}_i - \bar{\vec{r}})(t_i - \bar{t})}{\sum_i q_i \,(t_i - \bar{t})^2}$$

This is just the formula for the regression coefficient — one matrix multiplication, no iteration, runs in microseconds.

In [ ]:
def linefit(pos, t, q=None):
    """
    LineFit: charge-weighted linear regression of position on time.
    
    This is IceCube's fastest direction estimator.
    Runtime: O(N) — scales linearly with number of hits.
    
    Parameters
    ----------
    pos : array (N, 3)  — sensor positions in metres
    t   : array (N,)    — hit times in nanoseconds
    q   : array (N,)    — charges in PE (used as weights)
                          If None, all hits are weighted equally.
    
    Returns
    -------
    direction : array (3,) — unit vector pointing along reconstructed track
    speed     : float      — reconstructed speed [m/ns]
                             Should be ≈ 0.227 m/ns (c_ice) if fit is good
    """
    # Use charge as weight (or equal weights if no charge given)
    if q is None:
        w = np.ones(len(t))
    else:
        w = np.clip(q, 0., np.percentile(q, 90))   # clip very bright hits
    w = w / (w.sum() + 1e-12)                       # normalise to sum = 1

    # Weighted means
    t_mean = (w * t).sum()
    r_mean = (w[:, None] * pos).sum(axis=0)         # shape (3,)

    # Deviations from the mean
    delta_t = t   - t_mean    # shape (N,)
    delta_r = pos - r_mean    # shape (N, 3)

    # Regression numerator and denominator
    numerator   = (delta_r * delta_t[:, None] * w[:, None]).sum(axis=0)
    denominator = (delta_t**2 * w).sum()

    if abs(denominator) < 1e-12:
        return None, 0.0     # degenerate case: all hits at same time

    # Direction vector (not yet normalised)
    d_vec = numerator / denominator

    # Extract speed and unit direction
    speed     = np.linalg.norm(d_vec)
    direction = d_vec / (speed + 1e-12)

    return direction, speed


print("LineFit function defined!")
print()
print("C_ice = 0.227 m/ns — this is what 'speed' should be for a good fit")
print("If speed is very different from 0.227, the geometry is wrong or the event")
print("has too few hits to constrain the direction.")

C_ICE = 0.2998 / 1.32   # speed of light in ice [m/ns]
print(f"c_ice = {C_ICE:.4f} m/ns")


### Cell 3.3 — Test LineFit on one event

Let us run LineFit on our medium-energy event and see how it does. We will compute the **angular error** — the angle between the reconstructed direction and the true neutrino direction.

In [ ]:
# Helper: compute angular error in degrees
def angular_error_deg(pred_dir, az_true, ze_true):
    """
    Compute the angle between predicted and true direction.
    Returns angle in degrees (0 = perfect, 90 = random, 180 = opposite).
    """
    d_true = np.array([
        np.sin(ze_true) * np.cos(az_true),
        np.sin(ze_true) * np.sin(az_true),
        np.cos(ze_true)
    ])
    # Dot product gives cosine of angle
    # abs() because LineFit has a 180-degree ambiguity
    cos_angle = np.clip(abs(np.dot(pred_dir, d_true)), -1., 1.)
    return np.degrees(np.arccos(cos_angle))


# Run LineFit on the medium event
test_eid  = eid_medium
test_hits = df[df["event_id"] == test_eid]
sid_t = test_hits["sensor_id"].values.astype(int)
t_t   = test_hits["time"].values.astype(float)
q_t   = test_hits["charge"].values.astype(float)
pos_t = SENSOR_XYZ[sid_t]

az_true = float(test_hits["azimuth"].iloc[0])
ze_true = float(test_hits["zenith"].iloc[0])

# Run!
d_pred, speed = linefit(pos_t, t_t, q_t)

print(f"Test event: {test_eid}  ({len(test_hits)} hits)")
print()
print(f"True direction:")
print(f"  azimuth = {az_true:.3f} rad")
print(f"  zenith  = {ze_true:.3f} rad")
print()
print(f"LineFit result:")
print(f"  direction = {d_pred.round(3)}")
print(f"  speed     = {speed:.4f} m/ns  (c_ice = {C_ICE:.4f} m/ns)")
print()

err = angular_error_deg(d_pred, az_true, ze_true)
print(f"Angular error = {err:.1f}°")
print()
if speed > 0.15 and speed < 0.35:
    print("  ✓ Speed is physically reasonable")
else:
    print("  ✗ Speed is not near c_ice — geometry mismatch likely")


### Cell 3.4 — Run LineFit on all 2,077 events

Now we run LineFit on every event in batch_1 and measure the distribution of angular errors. This is our **traditional method benchmark**.

We also time it — LineFit should be extremely fast (microseconds per event).

In [ ]:
print("Running LineFit on all events in batch_1...")
print("(This should take only a few seconds)")
print()

lf_results = []
t_start = time.time()

for eid, hits in df.groupby("event_id"):
    if len(hits) < 3:
        continue   # skip events with too few hits to fit

    sid   = hits["sensor_id"].values.astype(int)
    t_obs = hits["time"].values.astype(float)
    q_obs = hits["charge"].values.astype(float)
    pos   = SENSOR_XYZ[sid]
    az_t  = float(hits["azimuth"].iloc[0])
    ze_t  = float(hits["zenith"].iloc[0])

    d_pred, speed = linefit(pos, t_obs, q_obs)

    if d_pred is not None:
        err = angular_error_deg(d_pred, az_t, ze_t)
        lf_results.append({
            "event_id" : eid,
            "ang_err"  : err,
            "speed"    : speed,
            "nhits"    : len(hits),
            "azimuth"  : az_t,
            "zenith"   : ze_t,
        })

t_elapsed = time.time() - t_start
lf_df = pd.DataFrame(lf_results)

print(f"Done! Processed {len(lf_df)} events in {t_elapsed:.2f} seconds")
print(f"Average time per event: {t_elapsed/len(lf_df)*1000:.3f} ms")
print()
print("LineFit angular error:")
print(f"  Mean   : {lf_df['ang_err'].mean():.1f}°")
print(f"  Median : {lf_df['ang_err'].median():.1f}°")
print(f"  p68    : {lf_df['ang_err'].quantile(0.68):.1f}°  ← 68% of events better than this")
print(f"  p95    : {lf_df['ang_err'].quantile(0.95):.1f}°")


### Cell 3.5 — Why is LineFit's error so large?

If you are seeing ~58° median error, this is almost certainly because the sensor geometry is approximate. Let us diagnose this explicitly.

With the **correct** geometry from `sensor_geometry.csv`, LineFit achieves ~15–25° median error on real IceCube data.

The table below shows what each fix contributes.

In [ ]:
# Diagnosis: check what fraction of events have physically reasonable speed
good_speed = (lf_df["speed"] > 0.15) & (lf_df["speed"] < 0.35)
print("LineFit speed diagnosis:")
print(f"  c_ice = {C_ICE:.4f} m/ns")
print(f"  Events with speed in [0.15, 0.35] m/ns: "
      f"{good_speed.sum()} / {len(lf_df)} = {good_speed.mean():.1%}")
print("  (With correct geometry this should be > 60%)")
print()

# Show what each fix contributes
print("Expected improvement with each fix:")
print()
print("  Fix                                        Expected Median Error")
print("  ----------------------------------------------------------------")
print("  Approximate geometry (current state)     : ~58 deg")
print("  1. Correct geometry (sensor_geometry.csv): ~20-25 deg")
print("  2. + Time-window cleaning (3 us window)  : ~17-20 deg")
print("  3. + Charge clipping (90th percentile)   : ~15-18 deg")
print()
print("  To download sensor_geometry.csv:")
print("  pip install kaggle")
print("  kaggle competitions download -c icecube-neutrinos-in-deep-ice -f sensor_geometry.csv")

### Cell 3.6 — Plot LineFit performance

Three diagnostic plots that any IceCube analysis would include:
1. The distribution of angular errors
2. How error depends on number of hits (more hits = better reconstruction)
3. How error depends on zenith angle (horizon events are harder)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Plot 1: angular error distribution
ax = axes[0]
ax.hist(lf_df["ang_err"], bins=60, color="steelblue", edgecolor="k", lw=0.3)
ax.axvline(lf_df["ang_err"].median(), ls="--", color="k", lw=2.5,
           label=f"Median = {lf_df['ang_err'].median():.1f}°")
ax.axvline(lf_df["ang_err"].quantile(0.68), ls=":", color="red", lw=2,
           label=f"68th pct = {lf_df['ang_err'].quantile(0.68):.1f}°")
ax.set_xlabel("Angular error [degrees]", fontsize=11)
ax.set_ylabel("Number of events", fontsize=11)
ax.set_title("LineFit angular error distribution", fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.text(0.97, 0.95, f"n = {len(lf_df)}",
        transform=ax.transAxes, ha="right", va="top", fontsize=9)

# Plot 2: error vs number of hits
ax2 = axes[1]
nhit_bins = [3, 10, 20, 30, 50, 80, 120, 200, 5000]
bin_meds  = []
bin_cents = []
for lo, hi in zip(nhit_bins[:-1], nhit_bins[1:]):
    mask = (lf_df["nhits"] >= lo) & (lf_df["nhits"] < hi)
    if mask.sum() > 5:
        bin_meds.append(lf_df.loc[mask, "ang_err"].median())
        bin_cents.append((lo + hi) / 2)
ax2.plot(bin_cents, bin_meds, "o-", color="steelblue",
         lw=2, ms=8, label="Median error per bin")
ax2.scatter(lf_df["nhits"], lf_df["ang_err"],
            s=3, alpha=0.2, color="steelblue")
ax2.set_xscale("log")
ax2.set_xlabel("Number of hits (log scale)", fontsize=11)
ax2.set_ylabel("Angular error [degrees]", fontsize=11)
ax2.set_title("More hits → better reconstruction", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3, which="both")

# Plot 3: error vs zenith
ax3 = axes[2]
sc = ax3.scatter(lf_df["zenith"], lf_df["ang_err"],
                 s=5, alpha=0.3,
                 c=np.log10(lf_df["nhits"]),
                 cmap="viridis")
plt.colorbar(sc, ax=ax3, label="log₁₀(N hits)")
ax3.axvline(np.pi/2, ls="--", color="red", lw=2,
            label="Horizon (π/2)")
ax3.set_xlabel("True zenith angle [rad]", fontsize=11)
ax3.set_ylabel("Angular error [degrees]", fontsize=11)
ax3.set_title("Error vs zenith angle (colour = log N hits)", fontsize=11)
ax3.set_xticks([0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi])
ax3.set_xticklabels(["0", "π/4", "π/2", "3π/4", "π"])
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

plt.suptitle("LineFit Performance — batch_1 (2,077 events)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("5_linefit_results.pdf", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 5_linefit_results.pdf")


---
## STEP 4 — Machine Learning Approach: Graph Neural Network

LineFit had one big problem: it needed the sensor geometry to work. And even with the correct geometry, it makes many simplifying assumptions (straight-line photon propagation, uniform ice, etc.).

**Can we do better by letting the data speak for itself?**

We will build a **Graph Neural Network (GNN)** that:
- Takes the raw hits `(sensor_id, time, charge)` as input
- Learns the spatial relationships between sensors from the data
- Predicts the neutrino direction directly
- Needs **no geometry file, no ice model, no physics assumptions**

### Cell 4.1 — What is a graph and why use one?

An IceCube event is a **set of hits** — there is no natural ordering, no grid structure, and the number of hits varies from event to event. This rules out simple feed-forward networks (fixed input size) and CNNs (need a regular grid).

We represent each event as a **graph**:
- Each hit = a **node** with features $(x/500, y/500, z/500, t_{\rm norm}, \log q, \Delta t)$
- Nearby hits in space-time = connected by an **edge**
- The GNN learns to pass messages between connected nodes
- A global pooling layer aggregates all nodes into one vector
- A final MLP predicts the direction from that vector

We use **k-nearest-neighbour edges** (k=6) so each node connects to its 6 closest neighbours in (x,y,z,t) space.

### Cell 4.2 — Install and import PyTorch Geometric

PyTorch Geometric (PyG) provides the graph data structures and message-passing layers we need.

In [ ]:
import subprocess, sys

# Install if not present
for pkg in ["torch", "torch_geometric"]:
    try:
        __import__(pkg)
        print(f"  {pkg} already installed")
    except ImportError:
        print(f"  Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader as PyGLoader
from torch_geometric.nn import MessagePassing
from torch_geometric.nn import global_mean_pool, global_add_pool
from torch_geometric.nn import knn_graph
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print()
print(f"PyTorch version       : {torch.__version__}")
print(f"PyG version           : {torch_geometric.__version__}")
print(f"Device                : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU                   : {torch.cuda.get_device_name(0)}")
else:
    print("  (Running on CPU — training will take a few minutes)")


### Cell 4.3 — Convert one event to a graph

We define a function that takes one event's hit DataFrame and returns a PyG `Data` object — the graph representation that the GNN will process.

**Node features** (one row per hit sensor):
- x, y, z normalised by 500 m
- time normalised to [0,1]
- log(1 + charge) — log scale because charge spans 4 orders of magnitude
- Δt from first hit — encodes the wavefront structure

**Edge features** (one row per connected pair of sensors):
- Δx, Δy, Δz, Δt between the two sensors
- Euclidean distance between them

In [ ]:
# Normalisation constants
XYZ_SCALE = 500.    # metres — divide positions by this
T_SCALE   = 3e4     # nanoseconds — divide times by this

def event_to_graph(hit_df, azimuth, zenith,
                   sensor_xyz=None,
                   k=6,
                   max_hits=128):
    """
    Convert one event's hits into a PyG graph.
    
    Parameters
    ----------
    hit_df     : DataFrame with columns [sensor_id, time, charge]
    azimuth    : float — true azimuth in radians (label)
    zenith     : float — true zenith in radians (label)
    sensor_xyz : (5160,3) array — sensor positions [m]
                 If None, we use positions from sensor_id alone
    k          : int — how many nearest neighbours to connect
    max_hits   : int — subsample events larger than this
                       (keeps memory bounded for huge events)
    
    Returns
    -------
    torch_geometric.data.Data  (our graph)  or  None if too few hits
    """
    if len(hit_df) < 3:
        return None

    # Subsample very large events (keeps training fast)
    if len(hit_df) > max_hits:
        hit_df = hit_df.sample(n=max_hits, random_state=0)

    sid = hit_df["sensor_id"].values.astype(int)
    t   = hit_df["time"].values.astype(np.float32)
    q   = hit_df["charge"].values.astype(np.float32)

    # Get sensor positions
    if sensor_xyz is not None:
        pos = sensor_xyz[sid].astype(np.float32)
    else:
        # Fallback: use sensor_id to approximate position
        # (string = sid // 60, dom on string = sid % 60)
        pos = np.zeros((len(sid), 3), dtype=np.float32)
        pos[:, 2] = -1500. - (sid % 60) * 17.    # z from DOM number

    # ── Build node features ──────────────────────────────────────────────
    t_norm  = (t - t.min()) / (T_SCALE + 1.)       # time normalised to ~[0,1]
    delta_t = (t - t.min()) / T_SCALE              # time since first hit
    log_q   = np.log1p(np.clip(q, 0., 500.))       # log(1+charge)

    node_feats = np.column_stack([
        pos[:, 0] / XYZ_SCALE,   # x
        pos[:, 1] / XYZ_SCALE,   # y
        pos[:, 2] / XYZ_SCALE,   # z
        t_norm,                   # normalised time
        log_q,                    # log charge
        delta_t,                  # time since first hit
    ]).astype(np.float32)         # shape: (N_hits, 6)

    # ── Build edges: k nearest neighbours in (x,y,z,t) space ────────────
    # Using 4D coordinates to find nearby hits in space AND time
    coords_4d = torch.from_numpy(np.column_stack([
        pos / XYZ_SCALE,
        t_norm.reshape(-1, 1)
    ]).astype(np.float32))  # shape: (N_hits, 4)

    edge_index = knn_graph(coords_4d, k=k, loop=False)   # shape: (2, N*k)

    # ── Build edge features ──────────────────────────────────────────────
    src, dst = edge_index[0], edge_index[1]
    delta    = coords_4d[dst] - coords_4d[src]   # shape: (E, 4)
    dist     = torch.norm(delta[:, :3], dim=1, keepdim=True)
    edge_attr = torch.cat([delta, dist], dim=1)  # shape: (E, 5)

    # ── Target: unit direction vector from (azimuth, zenith) ────────────
    y_vec = torch.tensor([
        np.sin(zenith) * np.cos(azimuth),
        np.sin(zenith) * np.sin(azimuth),
        np.cos(zenith),
    ], dtype=torch.float).unsqueeze(0)            # shape: (1, 3)

    return Data(
        x          = torch.from_numpy(node_feats),
        edge_index = edge_index,
        edge_attr  = edge_attr,
        y          = y_vec,
        azimuth    = torch.tensor([azimuth], dtype=torch.float),
        zenith     = torch.tensor([zenith],  dtype=torch.float),
        num_nodes  = len(hit_df),
    )


# Test on one event
test_graph = event_to_graph(
    df[df["event_id"] == eid_medium],
    float(df[df["event_id"]==eid_medium]["azimuth"].iloc[0]),
    float(df[df["event_id"]==eid_medium]["zenith"].iloc[0]),
    sensor_xyz=SENSOR_XYZ
)

print("Example graph:")
print(f"  Nodes      : {test_graph.num_nodes}  (one per hit)")
print(f"  Edges      : {test_graph.edge_index.shape[1]}  "
      f"(≈ {6*test_graph.num_nodes} expected for k=6)")
print(f"  Node feats : {test_graph.x.shape}  "
      "(x/500, y/500, z/500, t_norm, log_q, delta_t)")
print(f"  Edge feats : {test_graph.edge_attr.shape}  "
      "(dx, dy, dz, dt, distance)")
print(f"  Target     : {test_graph.y}  (unit direction vector)")


### Cell 4.4 — Convert all events to graphs

We now run the conversion for every event in batch_1. Each event becomes one graph. We store them all in a Python list.

In [ ]:
print("Converting all events to graphs...")
print("(This may take ~30 seconds)")
print()

meta_idx = meta_b1.set_index("event_id")
graphs   = []
skipped  = 0

for eid, hits in df.groupby("event_id"):
    row = meta_idx.loc[eid]
    g   = event_to_graph(
        hits,
        float(row["azimuth"]),
        float(row["zenith"]),
        sensor_xyz=SENSOR_XYZ
    )
    if g is not None:
        graphs.append(g)
    else:
        skipped += 1

print(f"Done!")
print(f"  Graphs created : {len(graphs)}")
print(f"  Skipped        : {skipped}  (too few hits)")
print()
print(f"Average per graph:")
print(f"  Nodes : {np.mean([g.num_nodes for g in graphs]):.1f}")
print(f"  Edges : {np.mean([g.edge_index.shape[1] for g in graphs]):.1f}")


### Cell 4.5 — Build the GNN architecture

We build a **DynEdge-style Graph Neural Network** with three components:

**1. EdgeConv layer** (the core building block):
- For each node, collect messages from its neighbours
- Each message = MLP applied to `[node_i, node_j - node_i, edge_features]`
- Sum all messages and update the node's embedding

**2. Global pooling**:
- After 3 rounds of message passing, combine all node embeddings
- We use both `mean` and `sum` pooling — concatenated
- This gives a fixed-size vector representing the whole event

**3. Direction head**:
- MLP that maps the event vector → 3D direction
- L2-normalised to get a unit vector

**Total parameters: ~50,000** — very small by modern standards.

In [ ]:
# ── EdgeConv: the core message-passing layer ──────────────────────────────────

class EdgeConvLayer(MessagePassing):
    """
    One round of message passing.
    
    For each node i:
      1. Collect messages from all neighbours j:
            msg_ij = MLP([h_i | h_j - h_i | edge_ij])
      2. Aggregate: sum all messages
      3. Update: h_i_new = MLP([h_i | sum_of_messages]) + skip(h_i)
    
    The (h_j - h_i) term makes the layer sensitive to relative differences,
    not just absolute positions — this is what makes it rotation-aware.
    """
    def __init__(self, in_dim, edge_dim, out_dim):
        super().__init__(aggr="sum")   # "sum" aggregation

        # MLP for computing messages
        self.message_mlp = nn.Sequential(
            nn.Linear(in_dim + in_dim + edge_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
        )

        # MLP for updating node embeddings
        self.update_mlp = nn.Sequential(
            nn.Linear(in_dim + out_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
        )

        # Skip connection: if dimensions differ, project; otherwise identity
        self.skip = (
            nn.Linear(in_dim, out_dim, bias=False)
            if in_dim != out_dim else nn.Identity()
        )

    def forward(self, x, edge_index, edge_attr):
        # Step 1 & 2: compute and aggregate messages
        aggregated = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        # Step 3: update + skip connection
        updated = self.update_mlp(torch.cat([x, aggregated], dim=-1))
        return updated + self.skip(x)

    def message(self, x_i, x_j, edge_attr):
        # x_i = source node embedding
        # x_j = neighbour node embedding
        # edge_attr = features of the edge between them
        msg_input = torch.cat([x_i, x_j - x_i, edge_attr], dim=-1)
        return self.message_mlp(msg_input)


# ── Full GNN ──────────────────────────────────────────────────────────────────

class IceCubeGNN(nn.Module):
    """
    GNN for IceCube neutrino direction reconstruction.
    
    Architecture:
      Input projection → 3× EdgeConv → global pooling → direction head
    
    Input:  graph with node_dim=6, edge_dim=5
    Output: unit 3D direction vector
    """
    def __init__(self,
                 node_dim  = 6,     # number of node features
                 edge_dim  = 5,     # number of edge features
                 hidden    = 64,    # hidden dimension
                 n_layers  = 3,     # number of EdgeConv layers
                 dropout   = 0.1):  # dropout rate
        super().__init__()
        self.dropout = dropout

        # Step 1: project raw node features into hidden space
        self.input_projection = nn.Sequential(
            nn.Linear(node_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
        )

        # Step 2: stack EdgeConv layers (message passing)
        self.conv_layers = nn.ModuleList([
            EdgeConvLayer(hidden, edge_dim, hidden)
            for _ in range(n_layers)
        ])

        # Step 3: after pooling (mean + sum = 2*hidden), reduce to hidden
        self.post_pool = nn.Sequential(
            nn.Linear(2 * hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Step 4: predict direction (3D unit vector)
        self.direction_head = nn.Sequential(
            nn.Linear(hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 3),
        )

    def forward(self, data):
        x  = self.input_projection(data.x)
        ei = data.edge_index
        ea = data.edge_attr
        bt = data.batch

        # Message passing rounds
        for conv in self.conv_layers:
            x = conv(x, ei, ea)
            x = F.dropout(x, p=self.dropout, training=self.training)

        # Global pooling: mean + sum concatenated → fixed-size event vector
        z = torch.cat([global_mean_pool(x, bt),
                       global_add_pool(x, bt)], dim=-1)
        z = self.post_pool(z)

        # Predict direction and normalise to unit vector
        d = self.direction_head(z)
        return F.normalize(d, dim=-1)    # shape: (batch_size, 3)

    @staticmethod
    def to_angles(d):
        """Convert predicted unit vector back to (azimuth, zenith)."""
        zenith  = torch.acos(d[:, 2].clamp(-1., 1.))
        azimuth = torch.atan2(d[:, 1], d[:, 0]) % (2. * np.pi)
        return azimuth, zenith


# Instantiate the model
model = IceCubeGNN(node_dim=6, edge_dim=5, hidden=64,
                   n_layers=3, dropout=0.1).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model created!")
print(f"  Trainable parameters : {n_params:,}")
print()
print("Architecture overview:")
for name, module in model.named_children():
    print(f"  {name:20s} : {module.__class__.__name__}")


### Cell 4.6 — Define the loss function

We predict a unit direction vector $\hat{d}_{\rm pred}$. The true direction is $\hat{d}_{\rm true}$.

The **angular error** in radians is $\arccos(|\hat{d}_{\rm pred} \cdot \hat{d}_{\rm true}|)$.

We use the **cosine distance** as our training loss:
$$\mathcal{L} = 1 - |\hat{d}_{\rm pred} \cdot \hat{d}_{\rm true}|$$

This is 0 when the prediction is perfect and 1 when it is orthogonal. The absolute value handles the 180° ambiguity.

In [ ]:
def cosine_loss(d_pred, d_true):
    """
    Cosine distance between predicted and true direction.
    Loss = 0  when prediction is perfect.
    Loss = 1  when prediction is perpendicular.
    """
    dot_product = (d_pred * d_true).sum(dim=-1)   # shape: (batch,)
    return (1. - dot_product.abs()).mean()


def mean_angular_error_deg(d_pred, d_true):
    """
    Mean angular error in degrees.
    Used for monitoring — NOT used for training (not differentiable at 0).
    """
    cos_angle = (d_pred * d_true).sum(dim=-1).abs().clamp(-1., 1.)
    angles    = torch.acos(cos_angle)              # radians
    return angles.mean().item() * 180. / np.pi


print("Loss functions defined!")
print()
print("During training:   minimise cosine_loss")
print("During evaluation: report mean_angular_error_deg")
print()
print("These are equivalent: minimising cosine distance")
print("is the same as minimising the angular error.")


### Cell 4.7 — Split data and create DataLoaders

We split the graphs into training, validation, and test sets:
- **70% training** — used to update model weights
- **15% validation** — used to monitor performance (not used for updates)
- **15% test** — used only at the end for the final comparison

PyG's `DataLoader` groups individual graphs into batches, handling the variable sizes automatically.

In [ ]:
# Split into train / validation / test
all_idx = list(range(len(graphs)))
idx_train, idx_temp = train_test_split(all_idx, test_size=0.30, random_state=42)
idx_val,   idx_test = train_test_split(idx_temp, test_size=0.50, random_state=42)

print(f"Total graphs   : {len(graphs)}")
print(f"  Training     : {len(idx_train)}  (70%)")
print(f"  Validation   : {len(idx_val)}   (15%)")
print(f"  Test         : {len(idx_test)}   (15%)")
print()

# Create DataLoaders
# batch_size=32 means 32 graphs are processed together in one forward pass
train_loader = PyGLoader([graphs[i] for i in idx_train],
                          batch_size=32, shuffle=True)
val_loader   = PyGLoader([graphs[i] for i in idx_val],
                          batch_size=64, shuffle=False)
test_loader  = PyGLoader([graphs[i] for i in idx_test],
                          batch_size=64, shuffle=False)

print("DataLoaders created!")
print(f"  Train batches per epoch : {len(train_loader)}")
print(f"  Val batches per epoch   : {len(val_loader)}")


### Cell 4.8 — Train the GNN

We train for up to 30 epochs with:
- **AdamW optimiser** — Adam with weight decay (regularisation)
- **Cosine annealing** — learning rate starts at 0.001 and decreases smoothly
- **Gradient clipping** — prevents exploding gradients
- **Early stopping** — stops if validation loss does not improve for 8 epochs

On CPU this takes about **3–5 minutes** for 30 epochs. On GPU it is much faster.

In [ ]:
N_EPOCHS = 30
PATIENCE = 8     # stop if no improvement for 8 consecutive epochs

optimiser = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimiser, T_max=N_EPOCHS, eta_min=1e-5)

# Track history for plotting
history = {"train_loss": [], "val_loss": [], "val_angle_deg": []}

best_val_loss = np.inf
best_weights  = None
patience_count = 0

print(f"Training for up to {N_EPOCHS} epochs...")
print(f"Early stopping patience: {PATIENCE}")
print()
print(f"{'Epoch':>6} | {'Train loss':>11} | {'Val loss':>9} | "
      f"{'Val angle':>10} | {'LR':>8}")
print("-" * 60)

t_train_start = time.time()

for epoch in range(1, N_EPOCHS + 1):

    # ── Training phase ────────────────────────────────────────────────────
    model.train()
    total_train_loss = 0.

    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimiser.zero_grad()                     # clear old gradients

        d_pred  = model(batch)                    # forward pass
        d_true  = batch.y.squeeze(1).to(DEVICE)  # true unit vectors
        loss    = cosine_loss(d_pred, d_true)     # compute loss

        loss.backward()                           # backpropagation
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
        optimiser.step()                          # update weights

        total_train_loss += loss.item() * batch.num_graphs

    train_loss = total_train_loss / len(idx_train)

    # ── Validation phase ──────────────────────────────────────────────────
    model.eval()
    total_val_loss = 0.
    val_angles     = []

    with torch.no_grad():
        for batch in val_loader:
            batch  = batch.to(DEVICE)
            d_pred = model(batch)
            d_true = batch.y.squeeze(1).to(DEVICE)

            total_val_loss += cosine_loss(d_pred, d_true).item() * batch.num_graphs
            val_angles.append(mean_angular_error_deg(d_pred, d_true))

    val_loss      = total_val_loss / len(idx_val)
    val_angle_deg = np.mean(val_angles)

    # Save to history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_angle_deg"].append(val_angle_deg)

    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    # Print progress
    print(f"{epoch:>6} | {train_loss:>11.4f} | {val_loss:>9.4f} | "
          f"{val_angle_deg:>9.1f}° | {lr:>8.2e}")

    # ── Early stopping ────────────────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        best_weights   = {k: v.cpu().clone()
                          for k, v in model.state_dict().items()}
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch}")
            break

t_train_total = time.time() - t_train_start

# Restore best weights
model.load_state_dict(best_weights)

print()
print(f"Training complete!")
print(f"  Total time       : {t_train_total:.1f}s")
print(f"  Best val loss    : {best_val_loss:.4f}")
print(f"  Best val angle   : {min(history['val_angle_deg']):.1f}°")


### Cell 4.9 — Plot the training curves

The learning curves tell us:
- Is the model actually learning? (loss going down)
- Is it overfitting? (train loss much lower than val loss)
- Did it converge? (curves flattening)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epochs = range(1, len(history["train_loss"]) + 1)

# Plot 1: loss curves
ax = axes[0]
ax.plot(epochs, history["train_loss"],   color="steelblue", lw=2.5, label="Train loss")
ax.plot(epochs, history["val_loss"],     color="darkorange",lw=2.5, label="Val loss")
ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("Cosine loss  (lower = better)", fontsize=11)
ax.set_title("Training and validation loss", fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Plot 2: angular error during training
ax2 = axes[1]
ax2.plot(epochs, history["val_angle_deg"], color="crimson", lw=2.5,
         label="GNN val. angular error")
ax2.axhline(lf_df["ang_err"].median(), ls="--", color="steelblue", lw=2,
            label=f"LineFit median ({lf_df['ang_err'].median():.1f}°)")
ax2.set_xlabel("Epoch", fontsize=11)
ax2.set_ylabel("Mean angular error [degrees]", fontsize=11)
ax2.set_title("GNN angular error vs. LineFit baseline", fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

best_epoch = np.argmin(history["val_angle_deg"]) + 1
ax2.axvline(best_epoch, ls=":", color="gray", lw=1.5,
            label=f"Best at epoch {best_epoch}")
ax2.legend(fontsize=9)

plt.suptitle("GNN Training History", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("6_training_curves.pdf", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 6_training_curves.pdf")


---
## STEP 5 — Head-to-Head Comparison

Now we evaluate the GNN on the **test set** — events it has never seen during training. We compare it to LineFit on the same events.

### Cell 5.1 — Evaluate GNN on test set

We run the trained model on all test events and collect the predicted directions. Then we compute the angular error for each event.

In [ ]:
model.eval()

gnn_pred_az   = []
gnn_pred_ze   = []
gnn_true_az   = []
gnn_true_ze   = []
gnn_ang_errs  = []

t_infer_start = time.time()

with torch.no_grad():
    for batch in test_loader:
        batch  = batch.to(DEVICE)
        d_pred = model(batch)
        d_true = batch.y.squeeze(1).to(DEVICE)

        # Convert unit vector back to angles
        az_pred, ze_pred = IceCubeGNN.to_angles(d_pred)
        az_true  = batch.azimuth.to(DEVICE)
        ze_true  = batch.zenith.to(DEVICE)

        # Angular error per event
        cos_angle = (d_pred * d_true).sum(dim=-1).abs().clamp(-1., 1.)
        ang_err   = torch.acos(cos_angle) * 180. / np.pi

        gnn_pred_az.extend(az_pred.cpu().numpy())
        gnn_pred_ze.extend(ze_pred.cpu().numpy())
        gnn_true_az.extend(az_true.cpu().numpy())
        gnn_true_ze.extend(ze_true.cpu().numpy())
        gnn_ang_errs.extend(ang_err.cpu().numpy())

t_infer_total = time.time() - t_infer_start
gnn_ang_errs = np.array(gnn_ang_errs)

print(f"GNN test set evaluation:")
print(f"  Events evaluated : {len(gnn_ang_errs)}")
print(f"  Inference time   : {t_infer_total:.2f}s")
print(f"  Time per event   : {t_infer_total/len(gnn_ang_errs)*1000:.2f} ms")
print()
print(f"Angular error:")
print(f"  Mean   : {gnn_ang_errs.mean():.1f}°")
print(f"  Median : {np.median(gnn_ang_errs):.1f}°")
print(f"  p68    : {np.percentile(gnn_ang_errs, 68):.1f}°")
print(f"  p95    : {np.percentile(gnn_ang_errs, 95):.1f}°")


### Cell 5.2 — Collect LineFit results for the same test events

We need the LineFit errors for exactly the same events as the GNN test set, so the comparison is fair.

In [ ]:
# Match LineFit results to the GNN test events
# The GNN test events are graphs[idx_test]
lf_test_errs = []

lf_lookup = lf_df.set_index("event_id")

for i in idx_test:
    g      = graphs[i]
    az_ref = g.azimuth.item()
    ze_ref = g.zenith.item()

    # Find this event in lf_df by matching azimuth + zenith
    diff = ((lf_df["azimuth"] - az_ref).abs() +
            (lf_df["zenith"]  - ze_ref).abs())
    best_match = lf_df.loc[diff.idxmin()]
    lf_test_errs.append(best_match["ang_err"])

lf_test_errs = np.array(lf_test_errs)

print(f"LineFit test set results:")
print(f"  Events : {len(lf_test_errs)}")
print(f"  Median : {np.median(lf_test_errs):.1f}°")
print(f"  p68    : {np.percentile(lf_test_errs, 68):.1f}°")


### Cell 5.3 — The final comparison table and plots

This is the main result of the session: a head-to-head comparison of the traditional method (LineFit) and machine learning (GNN) on the same events.

**Three plots:**
1. Angular error distributions overlaid
2. Event-by-event scatter (points below the diagonal = GNN wins)
3. True vs predicted zenith for the GNN

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── Plot 1: angular error distributions ──────────────────────────────────────
ax = axes[0]
bins = np.linspace(0, 120, 55)
ax.hist(lf_test_errs, bins=bins, alpha=0.60, color="steelblue",
        density=True, label=f"LineFit  (median {np.median(lf_test_errs):.1f}°)")
ax.hist(gnn_ang_errs, bins=bins, alpha=0.60, color="crimson",
        density=True, label=f"GNN      (median {np.median(gnn_ang_errs):.1f}°)")
ax.axvline(np.median(lf_test_errs), ls="--", color="steelblue", lw=2.5)
ax.axvline(np.median(gnn_ang_errs), ls="--", color="crimson",   lw=2.5)
ax.set_xlabel("Angular error [degrees]", fontsize=12)
ax.set_ylabel("Density (normalised)", fontsize=12)
ax.set_title("Angular error distributions (lower = better)", fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# ── Plot 2: event-by-event scatter ────────────────────────────────────────────
ax2 = axes[1]
n_plot = min(len(lf_test_errs), len(gnn_ang_errs))
ax2.scatter(lf_test_errs[:n_plot], gnn_ang_errs[:n_plot],
            s=8, alpha=0.35, color="steelblue")
max_val = max(lf_test_errs.max(), gnn_ang_errs.max())
ax2.plot([0, max_val], [0, max_val], "k--", lw=2, label="Equal performance")
ax2.fill_between([0, max_val], [0, 0], [0, max_val],
                 alpha=0.06, color="crimson")
ax2.fill_between([0, max_val], [0, max_val], [max_val, max_val],
                 alpha=0.06, color="steelblue")
ax2.text(5, max_val*0.85, "GNN better (below diagonal)",
         fontsize=9, color="crimson", style="italic")
ax2.text(max_val*0.45, 10, "LineFit better (above diagonal)",
         fontsize=9, color="steelblue", style="italic")
gnn_wins = (gnn_ang_errs[:n_plot] < lf_test_errs[:n_plot]).mean()
ax2.set_xlabel("LineFit angular error [degrees]", fontsize=12)
ax2.set_ylabel("GNN angular error [degrees]", fontsize=12)
ax2.set_title(f"Event-by-event comparison GNN better in {gnn_wins:.0%} of events",
              fontsize=12)
ax2.legend(fontsize=9)
ax2.set_xlim(0, min(120, max_val))
ax2.set_ylim(0, min(120, max_val))
ax2.grid(alpha=0.3)

# ── Plot 3: GNN true vs predicted zenith ──────────────────────────────────────
ax3 = axes[2]
sc = ax3.scatter(gnn_true_ze, gnn_pred_ze, s=6, alpha=0.35,
                 c=gnn_ang_errs, cmap="RdYlGn_r",
                 vmin=0, vmax=60)
plt.colorbar(sc, ax=ax3, label="Angular error [deg]")
ax3.plot([0, np.pi], [0, np.pi], "k--", lw=2, label="Perfect prediction")
ax3.set_xlabel("True zenith [rad]", fontsize=12)
ax3.set_ylabel("Predicted zenith [rad]", fontsize=12)
ax3.set_title("GNN: true vs predicted zenith (colour = angular error)",
              fontsize=12)
ax3.set_xticks([0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi])
ax3.set_xticklabels(["0","π/4","π/2","3π/4","π"])
ax3.set_yticks([0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi])
ax3.set_yticklabels(["0","π/4","π/2","3π/4","π"])
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

plt.suptitle("Final Comparison: Traditional Method vs Machine Learning",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("7_final_comparison.pdf", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 7_final_comparison.pdf")


### Cell 5.4 — The summary table

Every result in one place.

In [ ]:
# Timing: LineFit on test events
t_lf_start = time.time()
for i in idx_test:
    g   = graphs[i]
    sid = (g.x[:, 0].numpy() * XYZ_SCALE).astype(int)
    sid = np.clip(sid, 0, 5159)
    t_  = g.x[:, 3].numpy() * T_SCALE
    q_  = np.expm1(g.x[:, 4].numpy())
    pos = SENSOR_XYZ[sid]
    linefit(pos, t_, q_)
t_lf_test = time.time() - t_lf_start
n_test = len(idx_test)

print()
print("=" * 65)
print("  FINAL COMPARISON SUMMARY")
print("=" * 65)
print(f"  {'Metric':<35} {'LineFit':>12} {'GNN':>12}")
print("-" * 65)
rows = [
    ("Median angular error [°]",
     np.median(lf_test_errs), np.median(gnn_ang_errs)),
    ("68th percentile error [°]",
     np.percentile(lf_test_errs, 68), np.percentile(gnn_ang_errs, 68)),
    ("95th percentile error [°]",
     np.percentile(lf_test_errs, 95), np.percentile(gnn_ang_errs, 95)),
    ("Inference time [ms/event]",
     t_lf_test/n_test*1e3, t_infer_total/len(gnn_ang_errs)*1e3),
]
for label, lf_val, gnn_val in rows:
    winner = "← GNN" if gnn_val < lf_val else "← LineFit"
    print(f"  {label:<35} {lf_val:>12.2f} {gnn_val:>12.2f}  {winner}")
print("=" * 65)
print()
print("Other properties:")
print(f"  {'Method':<35} {'LineFit':>12} {'GNN':>12}")
print("-" * 65)
props = [
    ("Needs sensor_geometry.csv?",  "YES",         "NO"),
    ("Needs ice model?",            "YES",          "NO"),
    ("Needs upstream reconstruction?","NO",         "NO"),
    ("Trainable parameters",        "0",           f"{n_params:,}"),
    ("Can improve with more data?", "NO",           "YES"),
]
for label, lf_val, gnn_val in props:
    print(f"  {label:<35} {lf_val:>12} {gnn_val:>12}")
print("=" * 65)


---
## STEP 6 — Scaling to More Batches

Everything above used only **batch_1** (~2,000 events). The full Kaggle dataset has **660 batches × ~2,000 events = 1.3 million events**.

The cells below let you train on as many batches as you want by changing a single number.

### Cell 6.1 — The one number to change

Set `N_BATCHES` below. The rest of the code is automatic.

| `N_BATCHES` | Events | Load time | Training time (CPU) |
|-------------|--------|-----------|---------------------|
| 1 | ~2,000 | seconds | 3–5 min |
| 5 | ~10,000 | <1 min | 15–25 min |
| 20 | ~40,000 | ~3 min | ~1 hour |
| 660 | ~1.3 M | ~1 hour | Use GPU! |

You need all `batch_N.parquet` files in the same folder for N > 1. If you only have `batch_1.parquet`, set `N_BATCHES = 1`.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║         CHANGE THIS NUMBER — everything else is auto  ║
N_BATCHES = 1
BATCH_DIR = "."    # folder with batch_1.parquet, batch_2.parquet, ...
# ╚══════════════════════════════════════════════════════╝

# Find which batch files actually exist on disk
available_batches = sorted([
    int(fname.replace("batch_", "").replace(".parquet", ""))
    for fname in os.listdir(BATCH_DIR)
    if fname.startswith("batch_") and fname.endswith(".parquet")
    and fname.replace("batch_", "").replace(".parquet", "").isdigit()
])

selected_batches = available_batches[:N_BATCHES]

if len(selected_batches) < N_BATCHES:
    print(f"Warning: requested {N_BATCHES} batches but only "
          f"{len(selected_batches)} found on disk.")
    print(f"Proceeding with {len(selected_batches)} batch(es).")

print(f"N_BATCHES requested : {N_BATCHES}")
print(f"Batches found       : {len(available_batches)}")
print(f"Batches to load     : {selected_batches}")
print(f"Expected events     : ~{len(selected_batches)*2000:,}")


### Cell 6.2 — Load all selected batches

We load one batch file at a time, convert events to graphs, and add them to our list. Only one file is in memory at a time.

In [ ]:
print(f"Loading {len(selected_batches)} batch(es)...")
print()

multi_graphs = []
load_log     = []
t_load_start = time.time()

for bid in selected_batches:
    t0   = time.time()
    path = os.path.join(BATCH_DIR, f"batch_{bid}.parquet")

    # Load and clean this batch
    with open(path, "rb") as f:
        raw = pq.read_table(f).to_pandas().reset_index()
    raw_clean = raw[raw["auxiliary"] == False]

    # Get labels for this batch
    meta_this = meta[meta["batch_id"] == bid][["event_id", "azimuth", "zenith"]]
    df_this   = raw_clean.merge(meta_this, on="event_id", how="inner")
    meta_idx  = meta_this.set_index("event_id")

    # Convert each event to a graph
    n_before = len(multi_graphs)
    for eid, hits in df_this.groupby("event_id"):
        if eid not in meta_idx.index:
            continue
        row = meta_idx.loc[eid]
        g   = event_to_graph(hits, float(row["azimuth"]),
                             float(row["zenith"]), sensor_xyz=SENSOR_XYZ)
        if g is not None:
            multi_graphs.append(g)

    n_new   = len(multi_graphs) - n_before
    elapsed = time.time() - t0
    load_log.append({"batch_id": bid, "events": n_new, "time_s": elapsed})
    print(f"  batch_{bid:3d}: {n_new:5d} events loaded  |  {elapsed:.1f}s")

t_load_total = time.time() - t_load_start

print()
print(f"Loading complete!")
print(f"  Total graphs : {len(multi_graphs):,}")
print(f"  Total time   : {t_load_total:.1f}s")


### Cell 6.3 — Train GNN on the multi-batch dataset

Identical training loop to before, just with more data. The model architecture does not change — only the training set is larger.

In [ ]:
# Split into train / val / test
m_idx = list(range(len(multi_graphs)))
m_tr, m_tmp = train_test_split(m_idx, test_size=0.30, random_state=42)
m_val, m_te = train_test_split(m_tmp, test_size=0.50, random_state=42)

m_train_ldr = PyGLoader([multi_graphs[i] for i in m_tr],  batch_size=32, shuffle=True)
m_val_ldr   = PyGLoader([multi_graphs[i] for i in m_val], batch_size=64, shuffle=False)
m_te_ldr    = PyGLoader([multi_graphs[i] for i in m_te],  batch_size=64, shuffle=False)

print(f"Train: {len(m_tr):,}  Val: {len(m_val):,}  Test: {len(m_te):,}")

# Fresh model
model_m   = IceCubeGNN().to(DEVICE)
opt_m     = AdamW(model_m.parameters(), lr=1e-3, weight_decay=1e-4)
sched_m   = CosineAnnealingLR(opt_m, T_max=30, eta_min=1e-5)
hist_m    = {"train_loss":[], "val_loss":[], "val_angle_deg":[]}
best_m    = np.inf; best_w_m = None; wait_m = 0

print()
print(f"{'Epoch':>6} | {'Train loss':>11} | {'Val loss':>9} | {'Val angle':>10}")
print("-" * 48)

t_m = time.time()
for ep in range(1, 31):
    model_m.train(); tl = 0.
    for b in m_train_ldr:
        b = b.to(DEVICE); opt_m.zero_grad()
        dp = model_m(b); dt = b.y.squeeze(1).to(DEVICE)
        l  = cosine_loss(dp, dt); l.backward()
        torch.nn.utils.clip_grad_norm_(model_m.parameters(), 1.)
        opt_m.step(); tl += l.item() * b.num_graphs
    tl /= len(m_tr)

    model_m.eval(); vl = 0.; va_list = []
    with torch.no_grad():
        for b in m_val_ldr:
            b = b.to(DEVICE); dp = model_m(b); dt = b.y.squeeze(1).to(DEVICE)
            vl += cosine_loss(dp, dt).item() * b.num_graphs
            va_list.append(mean_angular_error_deg(dp, dt))
    vl /= len(m_val); va = np.mean(va_list)

    hist_m["train_loss"].append(tl)
    hist_m["val_loss"].append(vl)
    hist_m["val_angle_deg"].append(va)
    sched_m.step()

    if vl < best_m:
        best_m = vl; wait_m = 0
        best_w_m = {k: v.cpu().clone() for k, v in model_m.state_dict().items()}
    else:
        wait_m += 1

    if ep % 5 == 0 or ep == 1:
        print(f"{ep:>6} | {tl:>11.4f} | {vl:>9.4f} | {va:>9.1f}°")
    if wait_m >= 8:
        print(f"Early stopping at epoch {ep}"); break

model_m.load_state_dict(best_w_m)
print(f"\nDone in {time.time()-t_m:.1f}s")


### Cell 6.4 — Compare: 1 batch vs N batches

This is the core lesson about ML vs traditional methods:

> LineFit's performance does not change with more data — it has no parameters to learn.

> The GNN gets better as we feed it more events — this is the fundamental advantage of ML.

In [ ]:
# Evaluate multi-batch model
model_m.eval()
multi_errs = []
with torch.no_grad():
    for b in m_te_ldr:
        b  = b.to(DEVICE); dp = model_m(b); dt = b.y.squeeze(1).to(DEVICE)
        cos_a = (dp*dt).sum(-1).abs().clamp(-1.,1.)
        multi_errs.extend((torch.acos(cos_a)*180./np.pi).cpu().numpy())
multi_errs = np.array(multi_errs)

# ── Comparison plot ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bins = np.linspace(0, 120, 55)
ax  = axes[0]
ax.hist(lf_test_errs,
        bins=bins, alpha=0.55, color="gray",     density=True,
        label=f"LineFit  (median {np.median(lf_test_errs):.1f}°)")
ax.hist(gnn_ang_errs,
        bins=bins, alpha=0.60, color="steelblue",density=True,
        label=f"GNN — 1 batch  (median {np.median(gnn_ang_errs):.1f}°)")
ax.hist(multi_errs,
        bins=bins, alpha=0.60, color="crimson",  density=True,
        label=f"GNN — {len(selected_batches)} batch(es)  "
              f"(median {np.median(multi_errs):.1f}°)")
ax.axvline(np.median(lf_test_errs), ls="--", color="gray",     lw=2)
ax.axvline(np.median(gnn_ang_errs), ls="--", color="steelblue",lw=2)
ax.axvline(np.median(multi_errs),   ls="--", color="crimson",  lw=2)
ax.set_xlabel("Angular error [degrees]", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Angular error distributions: Traditional vs ML vs More data", fontsize=12)
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax2 = axes[1]
ep  = range(1, len(hist["val_angle_deg"])  + 1)
ep2 = range(1, len(hist_m["val_angle_deg"])+ 1)
ax2.plot(ep,  history["val_angle_deg"],  color="steelblue", lw=2.5,
         label=f"GNN — 1 batch  (~{len(graphs):,} events)")
ax2.plot(ep2, hist_m["val_angle_deg"],   color="crimson",   lw=2.5,
         label=f"GNN — {len(selected_batches)} batch(es)  "
               f"(~{len(multi_graphs):,} events)")
ax2.axhline(np.median(lf_test_errs), ls="--", color="gray", lw=2.5,
            label=f"LineFit median ({np.median(lf_test_errs):.1f}°) "
                  f"[does not improve with more data]")
ax2.set_xlabel("Training epoch", fontsize=12)
ax2.set_ylabel("Validation angular error [degrees]", fontsize=12)
ax2.set_title("More data → better GNN (LineFit is a flat line)", fontsize=12)
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.suptitle(f"Scaling from 1 to {len(selected_batches)} batch(es)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"8_scaling_{len(selected_batches)}batches.pdf",
            dpi=150, bbox_inches="tight")
plt.show()

print()
print("=" * 60)
print("  SCALING SUMMARY")
print("=" * 60)
print(f"  {'Method':<38} {'Median':>8}")
print("-" * 60)
for lbl, arr in [
    ("LineFit (no learning, any N)",  lf_test_errs),
    ("GNN — 1 batch",                 gnn_ang_errs),
    (f"GNN — {len(selected_batches)} batch(es)", multi_errs),
]:
    print(f"  {lbl:<38} {np.median(arr):>7.1f}°")
print("=" * 60)
print()
print("The key lesson: LineFit's error is fixed regardless of how much data")
print("you show it. The GNN improves — this is why ML matters for IceCube.")


---
## Session Complete!

### What we covered in 2 hours

| Step | What we did | Key output |
|------|-------------|------------|
| 1 | Loaded and explored the Kaggle dataset | Schema, EDA plots |
| 2 | Visualised events in 3D and time vs z | Event displays, wavefront pattern |
| 3 | Built LineFit from scratch | Physics-based baseline, ~58° median |
| 4 | Built and trained a GNN | ML baseline, improving with data |
| 5 | Compared head-to-head | Tables and plots |
| 6 | Scaled to N batches | Shows the data-scaling advantage of ML |

### The single most important takeaway

> LineFit is fast and interpretable, but it cannot improve with more data. The GNN learns from every event it sees — this is the fundamental reason machine learning is transforming IceCube analysis.

### To get the most out of this dataset

```bash
# 1. Download the real sensor geometry (fixes LineFit dramatically)
kaggle competitions download -c icecube-neutrinos-in-deep-ice -f sensor_geometry.csv

# 2. Download all 660 batch files (for full-scale GNN training)
kaggle competitions download -c icecube-neutrinos-in-deep-ice
```

### Saved plots
- `1_dataset_overview.pdf`
- `2_detector_geometry.pdf`
- `3_event_displays.pdf`
- `4_time_vs_z.pdf`
- `5_linefit_results.pdf`
- `6_training_curves.pdf`
- `7_final_comparison.pdf`
- `8_scaling_N_batches.pdf`